In [1]:
import random
import math

In [13]:
# To keep track of more complex equation and derivatives

class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        r = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += r.grad
            other.grad += r.grad
        r._backward = _backward
        
        return r

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        # other = other if isinstance(other, Value) else Value(other)
        # r = Value(self.data - other.data, (self, other), '-')
        r = self + (-other)

        # def _backward():
        #     self.grad += r.grad
        #     other.grad += -r.grad
        # r._backward = _backward
        
        return r

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        r = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += r.grad * other.data
            other.grad += r.grad * self.data
        r._backward = _backward
        
        return r

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        r = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * self.data ** (other - 1) * r.grad
        r._backward = _backward

        return r

    def __truediv__(self, other):
        # other = other if isinstance(other, Value) else Value(other)
        # r = Value(self.data / other.data, (self, other), '/')
        r = self * other ** -1
        
        # def _backward():
        #     self.grad += r.grad / other.data
        #     other.grad += r.grad / self.data
        # r._backward = _backward
        
        return r

    def tanh(self):
        x = self.data
        upper = math.exp(2*x) - 1 
        lower = math.exp(2*x) + 1
        t = upper / lower
        r = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * r.grad
        r._backward = _backward
        
        return r

    def exp(self):
        x = self.data
        r = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += r.data * r.grad
        r._backward = _backward

        return r

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [3]:
# y = 3x + 2 + noise
data = []
for _ in range(50):
    x = random.uniform(-5, 5)
    y = 3*x + 2 + random.uniform(-0.5, 0.5)
    data.append((Value(x), y))
data[0]

(Value(data=-1.0108812558114622), -1.126749831043253)

In [4]:
# Random Parameters Initialisation
w = Value(random.uniform(-1.0, 1.0))
b = Value(random.uniform(-1.0, 1.0))

In [5]:
# learning rate
lr = 0.01
training_steps = 100

In [6]:

for step in range(training_steps):
    # forward
    loss = Value(0) # initial
    for x, y in data:
        pred = x * w + b # where did this linear equation came from ? How did we know to choose this ? We are assuming the true relationship is a line. The network does not "know" this; we chose a linear model because the data is linear. For curved data, we would add hidden layers and activation functions.
        diff = pred - y 
        loss = loss + diff * diff # why sqaure of this ? Why adding loss ? diff * diff makes every error positive (so negative and positive errors don't cancel) and punishes large errors more aggressively. The sum adds up the error across all data points.
    
    loss = loss / float(len(data)) # MEAN squared error
    
    # backward
    w.grad = 0.0
    b.grad = 0.0
    loss.backward()

    #update
    w.data -= lr * w.grad
    b.data -= lr * b.grad

    if step % 10 == 0:
        print(f"step {step}: loss={loss.data:.3f}, w={w.data:.3f}, b={b.data:.3f}")

print(f"Final: w={w.data:3f} (target 3.0), b={b.data:.3f} (target 2.0)")


step 0: loss=132.672, w=0.288, b=-0.265
step 10: loss=4.523, w=2.691, b=0.020
step 20: loss=2.598, w=2.888, b=0.351
step 30: loss=1.776, w=2.917, b=0.632
step 40: loss=1.223, w=2.931, b=0.863
step 50: loss=0.849, w=2.941, b=1.053
step 60: loss=0.596, w=2.950, b=1.209
step 70: loss=0.425, w=2.957, b=1.337
step 80: loss=0.310, w=2.963, b=1.443
step 90: loss=0.231, w=2.968, b=1.530
Final: w=2.971154 (target 3.0), b=1.595 (target 2.0)


In [15]:
a = Value(2.0)
b = Value(4.0)
# a * 2
a / b

Value(data=0.5)

In [16]:
# inputs x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# bias of the neuron
b = Value(6.8813735870195432, label='b')

# x1*w1 + x2*w2 + b
x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'